# Поиск услуг: кандидатогенерация

До 50 объявлений на запрос. Основная метрика — macro Recall@50.

## 1. Настройки

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import importlib.metadata
import platform
import random
import shutil
import subprocess
from time import perf_counter

import bm25s
import torch
from sklearn.model_selection import train_test_split
from scipy.stats import bootstrap
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from IPython.display import display

DATA_DIR = Path("dataset")
SEED = 42
DEVICE_REQUEST = os.environ.get("AVITO_DEVICE", "auto").lower()
VALIDATION_SIZE = 0.2
SPLIT_PATH = Path("split.csv")
EXPERIMENTS_PATH = Path("experiments.csv")
BM25_CONFIG = {"k1": 1.2, "b": 0.75, "method": "lucene", "backend": "numpy"}
TOKEN_PATTERN = r"(?u)\b\w+\b"
RECALL_K = (1, 5, 10, 20, 50)
E1_VARIANTS = {
    "E1_title": {"title": 1.0, "description": 0.0},
    "E1_description": {"title": 0.0, "description": 1.0},
    "E1_title1_description1": {"title": 1.0, "description": 1.0},
    "E1_title2_description1": {"title": 2.0, "description": 1.0},
    "E1_title4_description1": {"title": 4.0, "description": 1.0},
}
TFIDF_VARIANTS = {
    "E2_word": {"analyzer": "word", "ngram_range": (1, 2), "token_pattern": TOKEN_PATTERN,
                "min_df": 2, "max_features": 300_000},
    "E2_char": {"analyzer": "char_wb", "ngram_range": (3, 5),
                "min_df": 2, "max_features": 200_000},
}
TFIDF_COMMON = {"lowercase": False, "norm": "l2", "sublinear_tf": True,
                "smooth_idf": True, "dtype": np.float32}
TFIDF_BATCH_SIZE = 256
TFIDF_THREADS = min(8, os.cpu_count() or 1)
E2_CONFIDENCE = 1 - 0.05 / len(TFIDF_VARIANTS)
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_CONFIDENCE = 1 - 0.05 / len(E1_VARIANTS)
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

/home/alexiva05/projects/pythonProjects/avito/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def select_device(requested: str) -> torch.device:
    if requested not in {"auto", "cpu", "cuda", "mps"}:
        raise ValueError("AVITO_DEVICE: auto, cpu, cuda или mps")
    if requested == "auto":
        requested = ("cuda" if torch.cuda.is_available() else
                     "mps" if torch.backends.mps.is_available() else "cpu")
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна: проверьте GPU, драйвер и сборку PyTorch")
    if requested == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS недоступен: требуется совместимый Mac и сборка PyTorch")
    return torch.device(requested)


def check_device(device: torch.device) -> None:
    # Проверяем вычисления и градиенты, а не только наличие устройства.
    x = torch.arange(12, dtype=torch.float32).reshape(3, 4).to(device)
    x.requires_grad_(True)
    loss = (x @ x.T).sum()
    loss.backward()
    expected_gradient = 2 * torch.arange(12, dtype=torch.float32).reshape(3, 4).sum(0)
    torch.testing.assert_close(x.grad.cpu(), expected_gradient.expand(3, 4))
    torch.testing.assert_close(loss.detach().cpu(), torch.tensor(1134.0))


DEVICE = select_device(DEVICE_REQUEST)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)
check_device(DEVICE)
print(f"PyTorch {torch.__version__}; устройство: {DEVICE}; вычисления и градиенты: OK")

PyTorch 2.14.0+cu130; устройство: cuda; вычисления и градиенты: OK


## 2. Данные

In [3]:
QUERY_FIELDS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]


def normalize_text(values: pd.Series) -> pd.Series:
    """Нижний регистр и нормализация пробелов."""
    return values.fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resources(root: Path) -> dict:
    try:
        gpu = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, check=True, timeout=10,
        ).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        gpu = "unavailable"
    return {
        "python": platform.python_version(), "platform": platform.platform(),
        "cpu_physical": psutil.cpu_count(logical=False), "cpu_logical": psutil.cpu_count(),
        "ram_total_bytes": psutil.virtual_memory().total,
        "ram_available_bytes": psutil.virtual_memory().available,
        "disk_free_bytes": shutil.disk_usage(root).free, "gpu": gpu,
        "versions": {name: importlib.metadata.version(name) for name in
                     ["pandas", "numpy", "pyarrow", "nbformat", "nbclient", "psutil", "bm25s", "scikit-learn", "torch"]},
    }

In [4]:
def audit_data(data_dir: Path) -> dict:
    tables, files, columns, examples = {}, [], [], {}
    for name in ["benchmark_items", "benchmark_queries", "train"]:
        path = data_dir / f"{name}.parquet"
        frame = pd.read_parquet(path)
        tables[name] = frame
        files.append({"file": path.name, "bytes": path.stat().st_size,
                      "rows": len(frame), "columns": len(frame.columns), "sha256": sha256(path)})
        schema = pq.read_schema(path)
        for column in frame:
            values = frame[column]
            columns.append({"table": name, "column": column, "type": str(schema.field(column).type),
                            "nulls": int(values.isna().sum()), "unique": int(values.nunique()),
                            "empty_strings": int(values.eq("").sum())})
        examples[name] = frame.head(3).map(
            lambda x: str(x)[:240] if pd.notna(x) else None)

    items, queries, train = (tables[n] for n in ["benchmark_items", "benchmark_queries", "train"])
    for frame, key, pattern in [(items, "item_id", r"[0-9a-f]{16}"),
                                (train, "item_id", r"[0-9a-f]{16}"),
                                (queries, "query_id", r".{16}")]:
        if not frame[key].str.fullmatch(pattern).fillna(False).all():
            raise ValueError(f"Invalid {key}")
    if not items.item_id.is_unique or not queries.query_id.is_unique:
        raise ValueError("Duplicate benchmark IDs require an explicit resolution")

    # В train нет query_id; группируем по тексту и контексту.
    identity = train[QUERY_FIELDS].copy()
    identity["search_query"] = normalize_text(identity.search_query)
    group_ids = identity.groupby(QUERY_FIELDS, dropna=False, sort=True).ngroup()
    pairs = pd.DataFrame({"group": group_ids, "item_id": train.item_id}).drop_duplicates()
    pairs["in_corpus"] = pairs.item_id.isin(items.item_id)
    positives = pairs.groupby("group").agg(positives=("item_id", "size"), in_corpus=("in_corpus", "sum"))
    contexts = identity.drop_duplicates().groupby("search_query", dropna=False).size()
    item_text = pd.DataFrame({field: normalize_text(items[field]) for field in
                              ["item_title_raw", "item_description_raw"]})
    title_counts = item_text.groupby("item_title_raw").size()
    summary = {
        "train_rows": len(train), "corpus_items": len(items), "benchmark_queries": len(queries),
        "train_has_query_id": "query_id" in train, "train_has_explicit_label": False,
        "label_assumption": "Each train row is a positive interaction; no explicit label is provided.",
        "query_context_groups": len(positives),
        "normalized_train_texts": int(identity.search_query.nunique()),
        "texts_with_multiple_contexts": int(contexts.gt(1).sum()),
        "duplicate_positive_rows_after_normalization": len(train) - len(pairs),
        "unique_positive_pairs": len(pairs),
        "positive_pairs_outside_corpus": int((~pairs.in_corpus).sum()),
        "unique_positive_items_outside_corpus": int(pairs.loc[~pairs.in_corpus, "item_id"].nunique()),
        "groups_without_retrievable_positive": int(positives.in_corpus.eq(0).sum()),
        "positive_count_distribution": positives.positives.describe(percentiles=[.5, .9, .99]).to_dict(),
        "macro_recall_corpus_ceiling": float((positives.in_corpus / positives.positives).mean()),
        "empty_train_query_rows": int(identity.search_query.eq("").sum()),
        "empty_benchmark_query_rows": int(normalize_text(queries.search_query).eq("").sum()),
        "benchmark_repeated_text_rows": int(normalize_text(queries.search_query).duplicated().sum()),
        "duplicate_item_text_rows": int(item_text.duplicated().sum()),
        "repeated_title_groups": int(title_counts.gt(1).sum()),
        "near_duplicate_status": "Not measured; repeated titles alone do not prove near duplication.",
        "event_time_columns": [c for c in train if any(t in c.lower() for t in ["timestamp", "date", "event_time"])],
        "dense_float32_384_bytes": len(items) * 384 * 4,
        "dense_float32_768_bytes": len(items) * 768 * 4,
    }
    return {
        "summary": summary, "files": pd.DataFrame(files),
        "columns": pd.DataFrame(columns), "examples": examples,
        "resources": resources(data_dir.parent),
    }

In [5]:
audit = audit_data(DATA_DIR)
summary = audit["summary"]
files = audit["files"]
files["Размер, МиБ"] = (files["bytes"] / 2**20).round(2)
display(files[["file", "rows", "columns", "Размер, МиБ"]].rename(
    columns={"file": "Файл", "rows": "Строки", "columns": "Поля"}))
display(files[["file", "sha256"]])

,Файл,Строки,Поля,"Размер, МиБ"
0,benchmark_items.parquet,189212,14,186.61
1,benchmark_queries.parquet,2452,6,0.12
2,train.parquet,497673,19,467.53


,file,sha256
0,benchmark_items.parquet,193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899
1,benchmark_queries.parquet,e49de4fb76f03979a4af96034817767d39ae5de9f94e254fed028243188dda06
2,train.parquet,e150ab7a5c98769f643b95ee3a02dc0f664b647facd69a7d280ec6d48ca554a7


`benchmark_items` — корпус объявлений, `benchmark_queries` — итоговые запросы,
`train` — пары запрос–объявление.

In [6]:
columns = audit["columns"]
column_names = {"column": "Поле", "type": "Тип Parquet", "nulls": "Пропуски",
                "unique": "Уникальные значения", "empty_strings": "Пустые строки"}
for table in files["file"].str.removesuffix(".parquet"):
    print(table)
    display(columns.loc[columns["table"].eq(table)].drop(columns="table").rename(columns=column_names))

benchmark_items


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
0,item_title_raw,string,0,119952,0
1,item_rating_reviews_count,double,11615,1170,0
2,item_rating,double,17631,6007,0
3,item_price,"decimal128(27, 15)",0,2949,0
4,item_microcat_id,int64,0,752,0
5,item_longitude,"decimal128(18, 15)",1,120004,0
6,item_location_id,int64,0,2877,0
7,item_latitude,"decimal128(17, 15)",1,119778,0
8,item_is_phone_hidden,bool,0,2,0
9,item_is_message_forbidden,bool,0,2,0


benchmark_queries


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
14,query_id,large_string,0,2452,0
15,search_query,large_string,0,2452,0
16,search_location_id,int64,0,273,0
17,search_is_delivery_search,int32,0,1,0
18,search_infm_params_text,large_string,0,132,1546
19,search_category,int64,0,2,0


train


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
20,search_query,string,0,74529,0
21,search_location_id,int64,0,2782,0
22,search_is_delivery_search,int32,0,2,0
23,search_infm_params_text,string,0,3724,163999
24,search_category,int64,0,4,0
25,item_title_raw,string,0,210029,0
26,item_rating_reviews_count,double,19211,1372,0
27,item_rating,double,28232,8832,0
28,item_price,"decimal128(27, 15)",0,2992,0
29,item_microcat_id,int64,0,212,0


### Примеры

Для отображения обрезаем значения до 240 символов.

In [7]:
for table, title in [("benchmark_queries", "Итоговый запрос"),
                     ("benchmark_items", "Объявление из корпуса"),
                     ("train", "Строка обучающей разметки")]:
    examples = audit["examples"][table]
    print(title)
    display(examples.iloc[0].rename("Пример значения").to_frame())

Итоговый запрос


,Пример значения
query_id,70DfDUpwjxB4lzFd
search_query,перевозки владикавказ тбилиси
search_location_id,649820
search_is_delivery_search,0
search_infm_params_text,
search_category,114


Объявление из корпуса


,Пример значения
item_title_raw,Ремонт/выкуп компьют. и ноутбуков с выездом на дом
item_rating_reviews_count,57.0
item_rating,5.0
item_price,500.000000000000000
item_microcat_id,2097573
item_longitude,42.047193961004901
item_location_id,631060
item_latitude,44.228180450924000
item_is_phone_hidden,False
item_is_message_forbidden,False


Строка обучающей разметки


,Пример значения
search_query,скупка телевизоров
search_location_id,652430
search_is_delivery_search,0
search_infm_params_text,
search_category,114
item_title_raw,Скупка б/у техники
item_rating_reviews_count,91.0
item_rating,4.989010989010989
item_price,1.000000000000000
item_microcat_id,2303374


В train нет `query_id`, явной метки релевантности и времени событий.
Каждую строку считаем позитивным взаимодействием.
Для BM25 используем заголовок и описание.

## 3. Запросы и повторы

Запрос определяем сочетанием пяти `search_*` полей. В тексте приводим буквы к нижнему
регистру и нормализуем пробелы. Одинаковый текст с разными фильтрами — разные запросы
для метрики, но одна группа для split.

In [8]:
query_counts = {
    "Строки train": summary["train_rows"],
    "Различные нормализованные тексты": summary["normalized_train_texts"],
    "Запросы с учётом контекста": summary["query_context_groups"],
    "Тексты с несколькими контекстами": summary["texts_with_multiple_contexts"],
    "Повторные позитивные пары": summary["duplicate_positive_rows_after_normalization"],
    "Уникальные позитивные пары": summary["unique_positive_pairs"],
}
display(pd.Series(query_counts, name="Количество").to_frame())
distribution = summary["positive_count_distribution"]
display(pd.Series({"Минимум": distribution["min"], "Медиана": distribution["50%"],
                   "90-й перцентиль": distribution["90%"], "99-й перцентиль": distribution["99%"],
                   "Максимум": distribution["max"], "Среднее": distribution["mean"]},
                  name="Позитивов на запрос с контекстом").to_frame().round(2))

,Количество
Строки train,497673
Различные нормализованные тексты,73904
Запросы с учётом контекста,354313
Тексты с несколькими контекстами,27894
Повторные позитивные пары,30644
Уникальные позитивные пары,467029


,Позитивов на запрос с контекстом
Минимум,1.00
Медиана,1.00
90-й перцентиль,2.00
99-й перцентиль,6.00
Максимум,278.00
Среднее,1.32


У большинства запросов один позитив, у некоторых — сотни.
Повторные пары запрос–объявление учитываем один раз.

## 4. Покрытие позитивов корпусом

Проверяем наличие размеченных `item_id` в корпусе. Позитивы вне корпуса
оставляем в знаменателе Recall.

In [9]:
total = summary["unique_positive_pairs"]
outside = summary["positive_pairs_outside_corpus"]
coverage = pd.DataFrame({
    "Позитивные пары": [total - outside, outside],
}, index=["Объявление есть в корпусе", "Объявления нет в корпусе"])
coverage["Доля пар, %"] = (100 * coverage["Позитивные пары"] / total).round(2)
display(coverage)
display(pd.Series({
    "Запросы с контекстом": summary["query_context_groups"],
    "Из них без доступных позитивов": summary["groups_without_retrievable_positive"],
}, name="Количество").to_frame())
print(f"Верхняя граница macro Recall по доступности корпуса: {summary['macro_recall_corpus_ceiling']:.2%}")

,Позитивные пары,"Доля пар, %"
Объявление есть в корпусе,29571,6.33
Объявления нет в корпусе,437458,93.67


,Количество
Запросы с контекстом,354313
Из них без доступных позитивов,327764


Верхняя граница macro Recall по доступности корпуса: 6.08%


Вне корпуса — 437 458 из 467 029 позитивных пар.
Верхняя граница macro Recall на всех train-группах — **6,08%**;
ограничение top-50 может снизить её ещё сильнее.

## 5. Дубли объявлений и пустые запросы

Сравниваем нормализованные заголовки и описания.

In [10]:
display(pd.Series({
    "Повторные строки title + description в корпусе": summary["duplicate_item_text_rows"],
    "Группы повторяющихся заголовков": summary["repeated_title_groups"],
    "Пустые запросы в train": summary["empty_train_query_rows"],
    "Пустые итоговые запросы": summary["empty_benchmark_query_rows"],
    "Повторные тексты итоговых запросов": summary["benchmark_repeated_text_rows"],
}, name="Количество").to_frame())

,Количество
Повторные строки title + description в корпусе,19097
Группы повторяющихся заголовков,15195
Пустые запросы в train,0
Пустые итоговые запросы,0
Повторные тексты итоговых запросов,1


Найдено 19 097 повторов заголовка и описания при разных item_id.
Сохраняем все ID: метрика различает такие объявления.
Benchmark ID уникальны, пустых поисковых текстов нет.

## 6. Ресурсы

In [11]:
resource_info = audit["resources"]
display(pd.Series({
    "Python": resource_info["python"],
    "CPU: физические ядра / потоки": f"{resource_info['cpu_physical']} / {resource_info['cpu_logical']}",
    "RAM всего, ГиБ": round(resource_info["ram_total_bytes"] / 2**30, 1),
    "RAM доступно, ГиБ": round(resource_info["ram_available_bytes"] / 2**30, 1),
    "Диск свободно, ГиБ": round(resource_info["disk_free_bytes"] / 2**30, 1),
    "GPU / память, МиБ / драйвер": resource_info["gpu"],
}, name="Значение").to_frame())
display(pd.DataFrame({
    "Размерность": [384, 768],
    "Embeddings float32, МиБ": [round(summary[f"dense_float32_{d}_bytes"] / 2**20, 1) for d in [384, 768]],
}))
display(pd.Series(resource_info["versions"], name="Версия библиотеки").to_frame())

,Значение
Python,3.12.3
CPU: физические ядра / потоки,16 / 32
"RAM всего, ГиБ",60.4
"RAM доступно, ГиБ",42.3
"Диск свободно, ГиБ",1307.5
"GPU / память, МиБ / драйвер","NVIDIA GeForce RTX 5090, 32607 MiB, 595.91.07"


,Размерность,"Embeddings float32, МиБ"
0,384,277.2
1,768,554.3


,Версия библиотеки
pandas,3.0.6
numpy,2.5.3
pyarrow,25.0.1
nbformat,5.11.1
nbclient,0.11.0
psutil,7.2.2
bm25s,0.3.11
scikit-learn,1.9.1
torch,2.14.0


Embeddings float32 займут 277–554 МиБ при размерности 384–768,
без учёта модели, батчей и индекса.

## 7. Валидация

Откладываем 20% уникальных нормализованных текстов, seed 42.
Все контексты и позитивы одного текста остаются вместе. Повторные пары удаляем.
Временной split невозможен: в данных нет времени событий.

Словарь BM25 строим по корпусу объявлений. Validation labels используются только для оценки.

In [12]:
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet").sort_values("item_id").reset_index(drop=True)
train = pd.read_parquet(DATA_DIR / "train.parquet", columns=QUERY_FIELDS + ["item_id"])
train["search_query"] = normalize_text(train["search_query"])
texts = np.sort(train["search_query"].unique())
train_texts, validation_texts = train_test_split(
    texts, test_size=VALIDATION_SIZE, random_state=SEED, shuffle=True)
validation_texts = set(validation_texts)
text_ids = {text: hashlib.sha256(text.encode("utf-8")).hexdigest() for text in texts}
split = pd.DataFrame({
    "text_id": [text_ids[text] for text in texts],
    "split": ["validation" if text in validation_texts else "train" for text in texts],
}).sort_values("text_id").reset_index(drop=True)
assert split["text_id"].is_unique
if SPLIT_PATH.exists():
    pd.testing.assert_frame_equal(pd.read_csv(SPLIT_PATH), split)
else:
    split.to_csv(SPLIT_PATH, index=False)
SPLIT_HASH = sha256(SPLIT_PATH)

train["group_id"] = train.groupby(QUERY_FIELDS, dropna=False, sort=True).ngroup()
train["split"] = np.where(train["search_query"].isin(validation_texts), "validation", "train")
assert train.groupby("group_id")["split"].nunique().eq(1).all()
assert set(train_texts).isdisjoint(validation_texts)
assert len(train_texts) + len(validation_texts) == len(texts)
pairs = train[["group_id", "item_id"]].drop_duplicates()
queries = train.drop_duplicates("group_id").set_index("group_id")[["search_query", "split"]]
queries["positives"] = pairs.groupby("group_id")["item_id"].agg(frozenset)
corpus_ids = frozenset(items["item_id"])
queries["available"] = queries["positives"].map(lambda values: values & corpus_ids)
queries["n_positives"] = queries["positives"].map(len)
queries["n_available"] = queries["available"].map(len)
validation = queries.loc[queries["split"].eq("validation")].copy()
assert queries["n_positives"].gt(0).all()
assert len(pairs) == summary["unique_positive_pairs"]
display(queries.groupby("split").agg(
    texts=("search_query", "nunique"), groups=("search_query", "size"),
    positive_pairs=("n_positives", "sum"), available_pairs=("n_available", "sum")))
print(f"SHA-256 split: {SPLIT_HASH}")
print(f"Validation: {validation.n_available.eq(0).sum():,} групп без доступных позитивов")

,texts,groups,positive_pairs,available_pairs
split,,,,
train,59123,285932,379842,23844
validation,14781,68381,87187,5727


SHA-256 split: bc175ad59e9f1cd311579a22c1c23682aa27fbac75a03c6f6141fe4ee2e0732e
Validation: 63,229 групп без доступных позитивов


### Пересечение объявлений

Одно объявление может быть позитивом разных запросов в обеих частях split.
Считаем общие item_id и совпадения текстов среди доступных позитивов.
Объявления с одинаковыми текстами сохраняем под исходными ID.

In [13]:
train_positive_ids = frozenset(pairs.loc[pairs.group_id.isin(queries.index[queries.split.eq("train")]), "item_id"])
val_positive_ids = frozenset(pairs.loc[pairs.group_id.isin(validation.index), "item_id"])
item_signatures = pd.Series(list(zip(
    normalize_text(items["item_title_raw"]), normalize_text(items["item_description_raw"])
)), index=items.item_id)
train_signatures = set(item_signatures.loc[item_signatures.index.isin(train_positive_ids)])
val_corpus_signatures = item_signatures.loc[item_signatures.index.isin(val_positive_ids)]
display(pd.Series({
    "Общие позитивные item_id между частями": len(train_positive_ids & val_positive_ids),
    "Validation item_id в корпусе": len(val_corpus_signatures),
    "Из них с текстом среди train-позитивов корпуса": int(val_corpus_signatures.isin(train_signatures).sum()),
}, name="Количество").to_frame())
del item_signatures, train_signatures, val_corpus_signatures

,Количество
Общие позитивные item_id между частями,25513
Validation item_id в корпусе,4809
Из них с текстом среди train-позитивов корпуса,2269


## 8. Метрики

Macro Recall@K усредняем по запросам с контекстом. Позитивы вне корпуса остаются
в знаменателе. Дубликаты выдачи удаляем до отсечения top-K.
Запросы без разметки исключаем из среднего и считаем отдельно.

Дополнительно считаем HitRate@50 и Recall@50 по доступным позитивам —
только среди групп, где есть хотя бы один такой позитив.

In [14]:
def query_metrics(positives, candidates, ks=RECALL_K):
    positives = set(positives)
    unique = list(dict.fromkeys(candidates))
    recalls = {f"Recall@{k}": len(positives.intersection(unique[:k])) / len(positives)
               if positives else np.nan for k in ks}
    recalls["HitRate@50"] = float(bool(positives.intersection(unique[:50]))) if positives else np.nan
    return recalls


def evaluate(query_table, predictions):
    rows = []
    for query in query_table.itertuples():
        candidates = predictions[query.search_query]
        values = query_metrics(query.positives, candidates)
        values["available_Recall@50"] = query_metrics(query.available, candidates)["Recall@50"]
        rows.append(values)
    return pd.DataFrame(rows, index=query_table.index)


assert query_metrics({"a", "b"}, ["a", "a", "b"], (1, 2))["Recall@2"] == 1
assert query_metrics({"a", "outside"}, ["a"])["Recall@50"] == 0.5
assert query_metrics({"outside"}, ["a"])["Recall@50"] == 0
assert query_metrics({"a"}, [])["HitRate@50"] == 0
assert np.isnan(query_metrics(set(), ["a"])["Recall@50"])
# Macro: равный вес запросов, независимо от числа позитивов.
assert np.mean([query_metrics({"a", "b"}, ["a"])["Recall@50"],
                query_metrics({"c"}, ["c"])["Recall@50"]]) == 0.75
print("Метрики: проверки пройдены")

Метрики: проверки пройдены


## 9. E0 — BM25

BM25 по заголовку и описанию: `bm25s`, вариант `lucene`, `k1=1.2`, `b=0.75`, float32.
Параметры без подбора.

Нижний регистр, нормализация пробелов, токены `\w+`. Сохраняем одиночные буквы,
числа, стоп-слова и «ё». Без стемминга и лемматизации.

При равных оценках сортируем по item_id. Нулевые оценки не включаем.
Один текст ищем один раз, каждый контекст оцениваем отдельно.

In [15]:
def tokenize(texts):
    return bm25s.tokenize(texts, lower=False, stopwords=[], token_pattern=TOKEN_PATTERN,
                          return_ids=False, show_progress=False)


def bm25_scores(model, tokens):
    # get_scores() в bm25s не принимает пустой список токенов.
    return model.get_scores_from_ids(model.get_tokens_ids(tokens))


def top_k(scores, k=50):
    if k <= 0:
        return np.empty(0, dtype=np.int64)
    selected = np.flatnonzero(scores > 0)
    if len(selected) > k:
        threshold = np.partition(scores[selected], -k)[-k]
        higher = selected[scores[selected] > threshold]
        tied = selected[scores[selected] == threshold][:k - len(higher)]
        selected = np.concatenate([higher, tied])
    return selected[np.lexsort((selected, -scores[selected]))]


assert tokenize(["ё  Ё  A-15 и"])[0] == ["ё", "Ё", "A", "15", "и"]
assert top_k(np.array([2., 1., 1., 0.]), 2).tolist() == [0, 1]
assert top_k(np.zeros(3)).tolist() == []
assert top_k(np.ones(60), 50).tolist() == list(range(50))
rng = np.random.default_rng(SEED)
for size in (1, 20, 100):
    scores = rng.integers(0, 5, size).astype(np.float32)
    expected = sorted(np.flatnonzero(scores > 0), key=lambda i: (-scores[i], i))[:50]
    assert top_k(scores).tolist() == expected

small_ids = np.array(["0000000000000001", "0000000000000002", "0000000000000003"])
small = bm25s.BM25(**BM25_CONFIG)
small.index(tokenize(["ремонт авто", "ремонт авто", "уборка кухни"]), show_progress=False)
assert small_ids[top_k(bm25_scores(small, ["ремонт"]))].tolist() == small_ids[:2].tolist()
assert top_k(bm25_scores(small, ["несуществующийтокен"])).tolist() == []
assert top_k(bm25_scores(small, [])).tolist() == []
# Для одинаковых документов длины 2 оценка каждого совпадения известна из формулы.
expected_score = np.log(1 + (3 - 2 + 0.5) / (2 + 0.5)) / (1 + BM25_CONFIG["k1"])
np.testing.assert_allclose(bm25_scores(small, ["ремонт"])[:2], expected_score, rtol=1e-6)
print("BM25, top-k и соответствие item_id: проверки пройдены")

BM25, top-k и соответствие item_id: проверки пройдены


In [16]:
start = perf_counter()
corpus_text = normalize_text(items["item_title_raw"].fillna("") + " " + items["item_description_raw"].fillna(""))
corpus_tokens = tokenize(corpus_text.tolist())
retriever = bm25s.BM25(**BM25_CONFIG)
retriever.index(corpus_tokens, show_progress=False)
index_seconds = perf_counter() - start
item_ids = items["item_id"].to_numpy()
assert items["item_id"].is_unique and items["item_id"].is_monotonic_increasing
assert retriever.scores["num_docs"] == len(item_ids)
index_bytes = sum(value.nbytes for value in retriever.scores.values() if isinstance(value, np.ndarray))
del corpus_tokens
_ = gc.collect()
print(f"Индексация: {index_seconds:.2f} с; массивы индекса: {index_bytes / 2**20:.1f} МиБ")

Индексация: 21.61 с; массивы индекса: 183.3 МиБ


In [17]:
predictions, query_seconds = {}, []
val_texts = sorted(validation["search_query"].unique())
start = perf_counter()
val_tokens = tokenize(val_texts)
for text, tokens in zip(val_texts, val_tokens):
    query_start = perf_counter()
    positions = top_k(bm25_scores(retriever, tokens), max(RECALL_K))
    predictions[text] = item_ids[positions].tolist()
    query_seconds.append(perf_counter() - query_start)
search_seconds = perf_counter() - start
assert set(predictions) == set(val_texts)
assert all(len(ids) <= 50 and len(ids) == len(set(ids)) and set(ids) <= corpus_ids
           for ids in predictions.values())
per_query = evaluate(validation, predictions)
metrics = per_query.mean()
assert per_query["Recall@50"].notna().all()
assert per_query[[f"Recall@{k}" for k in RECALL_K]].diff(axis=1).iloc[:, 1:].ge(0).all().all()
ceiling = (validation["n_available"].clip(upper=50) / validation["n_positives"]).mean()
assert metrics["Recall@50"] <= ceiling + 1e-12
display(metrics.rename("Значение").to_frame().round(6))
display(pd.Series({
    "Validation-группы": len(validation),
    "Группы без разметки": int(validation.n_positives.eq(0).sum()),
    "Группы с доступными позитивами": int(validation.n_available.gt(0).sum()),
    "Уникальные тексты для поиска": len(val_texts),
    "Тексты без кандидатов": sum(not values for values in predictions.values()),
    "Верхняя граница Recall@50": ceiling,
    "Индексация, с": index_seconds,
    "Поиск с токенизацией, с": search_seconds,
    "Поиск без токенизации, p50 мс": float(np.percentile(query_seconds, 50) * 1000),
    "Поиск без токенизации, p95 мс": float(np.percentile(query_seconds, 95) * 1000),
    "Массивы индекса, МиБ": index_bytes / 2**20,
    "RSS процесса после поиска, МиБ": psutil.Process().memory_info().rss / 2**20,
}, name="Значение").to_frame())

,Значение
Recall@1,0.001626
Recall@5,0.005165
Recall@10,0.008089
Recall@20,0.012069
Recall@50,0.019168
HitRate@50,0.023369
available_Recall@50,0.297620


,Значение
Validation-группы,68381.000000
Группы без разметки,0.000000
Группы с доступными позитивами,5152.000000
Уникальные тексты для поиска,14781.000000
Тексты без кандидатов,45.000000
Верхняя граница Recall@50,0.061306
"Индексация, с",21.608033
"Поиск с токенизацией, с",3.516701
"Поиск без токенизации, p50 мс",0.175944
"Поиск без токенизации, p95 мс",0.547823


Поиск последовательный, на CPU. Индексация включает подготовку текста и токенизацию.
RSS показывает память всего процесса после поиска, а не пиковое потребление индекса.

In [18]:
experiment = {
    "experiment": "E0", "status": "completed", "config": json.dumps(BM25_CONFIG, sort_keys=True),
    "token_pattern": TOKEN_PATTERN, "fields": "title + description", "seed": SEED,
    "validation_size": VALIDATION_SIZE, "split_sha256": SPLIT_HASH,
    "corpus_sha256": sha256(DATA_DIR / "benchmark_items.parquet"),
    "train_sha256": sha256(DATA_DIR / "train.parquet"),
    "bm25s_version": importlib.metadata.version("bm25s"), "device": "cpu",
    **{name: float(value) for name, value in metrics.items()}, "delta_Recall@50": 0.0,
    "index_seconds": index_seconds, "search_seconds": search_seconds,
    "index_MiB": index_bytes / 2**20, "validation_groups": len(validation),
    "conclusion": f"Recall@50={metrics['Recall@50']:.4%}; по доступным позитивам={metrics['available_Recall@50']:.2%}",
}
experiments = pd.DataFrame([experiment])
# Повторный запуск обновляет E0; результаты других экспериментов сохраняются.
if EXPERIMENTS_PATH.exists():
    previous = pd.read_csv(EXPERIMENTS_PATH)
    experiments = pd.concat([previous.loc[previous.experiment.ne("E0")], experiments], ignore_index=True)
experiments.to_csv(EXPERIMENTS_PATH, index=False)
display(experiments[["experiment", "status", "Recall@1", "Recall@5", "Recall@10", "Recall@20", "Recall@50"]])

,experiment,status,Recall@1,Recall@5,Recall@10,Recall@20,Recall@50
0,E1_title,completed,0.001385,0.004453,0.006679,0.009545,0.014627
1,E1_description,completed,0.001275,0.004290,0.007144,0.010456,0.016294
2,E1_title1_description1,completed,0.001782,0.005579,0.008227,0.012068,0.018960
3,E1_title2_description1,completed,0.001769,0.005324,0.008113,0.011961,0.018315
4,E1_title4_description1,completed,0.001545,0.005050,0.007639,0.011418,0.017560
5,E2_word,completed,0.000755,0.002981,0.005523,0.009084,0.015504
6,E2_char,completed,0.000714,0.002582,0.004389,0.007419,0.013516
7,E0,completed,0.001626,0.005165,0.008089,0.012069,0.019168


BM25: **Recall@50 = 1,92%** по 68 381 validation-группе.
У 63 229 групп нет позитивов в корпусе; верхняя граница Recall@50 — **6,13%**.
По доступным позитивам среди 5 152 групп Recall@50 — **29,76%**.
У 45 текстов нет совпадений со словарём корпуса.

## 10. E1 — вес заголовка

Гипотеза: совпадения в заголовке полезнее совпадений в длинном описании.
Сравниваем два поля отдельно и `w × BM25(title) + BM25(description)` при `w ∈ {1, 2, 4}`.
Это сумма оценок двух независимых индексов, не BM25F.

В каждом поле свои IDF и нормализация длины. Корпус, split, токенизация, `k1` и `b` те же, что в E0.
Нулевой вес исключает поле; равенства оценок разрешаем по item_id.

In [19]:
def weighted_bm25_scores(models, tokens, weights):
    scores = np.zeros(models["title"].scores["num_docs"], dtype=np.float32)
    for field, weight in weights.items():
        if weight:
            scores += weight * bm25_scores(models[field], tokens)
    return scores


def cluster_delta_interval(baseline, candidate, text_groups, confidence=BOOTSTRAP_CONFIDENCE):
    clustered = pd.DataFrame({
        "delta": np.asarray(candidate) - np.asarray(baseline),
        "text": np.asarray(text_groups),
    }).groupby("text", sort=True)["delta"].agg(["sum", "size"])
    # Пересэмплируем тексты, но сохраняем равный вес контекстных запросов в macro Recall.
    result = bootstrap(
        (clustered["sum"].to_numpy(), clustered["size"].to_numpy()),
        lambda totals, counts, axis: totals.sum(axis=axis) / counts.sum(axis=axis),
        paired=True, vectorized=True, n_resamples=BOOTSTRAP_RESAMPLES, batch=64,
        confidence_level=confidence, method="percentile", rng=np.random.default_rng(SEED),
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


small_description = bm25s.BM25(**BM25_CONFIG)
small_description.index(tokenize(["уборка кухни", "уборка кухни", "ремонт авто"]), show_progress=False)
small_fields = {"title": small, "description": small_description}
np.testing.assert_array_equal(
    weighted_bm25_scores(small_fields, ["ремонт"], E1_VARIANTS["E1_title"]),
    bm25_scores(small, ["ремонт"]))
np.testing.assert_array_equal(
    weighted_bm25_scores(small_fields, ["ремонт"], E1_VARIANTS["E1_description"]),
    bm25_scores(small_description, ["ремонт"]))
combined = weighted_bm25_scores(small_fields, ["ремонт"], E1_VARIANTS["E1_title2_description1"])
np.testing.assert_allclose(combined, 2 * bm25_scores(small, ["ремонт"]) + bm25_scores(small_description, ["ремонт"]))
assert small_ids[top_k(combined, 1)].tolist() == ["0000000000000003"]
assert top_k(weighted_bm25_scores(small_fields, [], E1_VARIANTS["E1_title1_description1"])).tolist() == []
assert cluster_delta_interval([0, 0, 0], [.25, .25, .25], ["a", "a", "b"]) == (.25, .25)
assert cluster_delta_interval([.5, 1, 0], [.5, 1, 0], ["a", "a", "b"]) == (0, 0)
print("Оценки полей, соответствие ID и bootstrap: проверки пройдены")

Оценки полей, соответствие ID и bootstrap: проверки пройдены


In [20]:
field_models, field_index_seconds, field_index_bytes = {}, {}, {}
for field, column in [("title", "item_title_raw"), ("description", "item_description_raw")]:
    start = perf_counter()
    field_tokens = tokenize(normalize_text(items[column]).tolist())
    model = bm25s.BM25(**BM25_CONFIG)
    model.index(field_tokens, show_progress=False)
    assert model.scores["num_docs"] == len(item_ids)
    field_models[field] = model
    field_index_seconds[field] = perf_counter() - start
    field_index_bytes[field] = sum(value.nbytes for value in model.scores.values() if isinstance(value, np.ndarray))
    del field_tokens
_ = gc.collect()
display(pd.DataFrame({
    "Индексация, с": field_index_seconds,
    "Массивы индекса, МиБ": {field: size / 2**20 for field, size in field_index_bytes.items()},
}))

,"Индексация, с","Массивы индекса, МиБ"
title,1.508178,6.860123
description,21.285713,181.266212


Интервалы изменения Recall@50: парный bootstrap по текстам, 2 000 повторов.
Для пяти сравнений с E0 используем интервалы 99% (поправка Бонферрони).
Выбор лучшего веса и оценка сделаны на одном validation split.

In [21]:
e1_rows = []
e1_per_query = {"E0": per_query}
best_e1_experiment, best_e1_recall = None, -np.inf
best_e1_predictions = None
for name, weights in E1_VARIANTS.items():
    variant_predictions, variant_query_seconds = {}, []
    start = perf_counter()
    variant_tokens = tokenize(val_texts)
    for text, tokens in zip(val_texts, variant_tokens):
        query_start = perf_counter()
        positions = top_k(weighted_bm25_scores(field_models, tokens, weights), max(RECALL_K))
        variant_predictions[text] = item_ids[positions].tolist()
        variant_query_seconds.append(perf_counter() - query_start)
    variant_search_seconds = perf_counter() - start
    assert set(variant_predictions) == set(val_texts)
    assert all(len(ids) <= 50 and len(ids) == len(set(ids)) and set(ids) <= corpus_ids
               for ids in variant_predictions.values())
    variant_per_query = evaluate(validation, variant_predictions)
    assert variant_per_query.index.equals(per_query.index)
    assert variant_per_query["Recall@50"].notna().all()
    assert variant_per_query[[f"Recall@{k}" for k in RECALL_K]].diff(axis=1).iloc[:, 1:].ge(0).all().all()
    variant_metrics = variant_per_query.mean()
    assert variant_metrics["Recall@50"] <= ceiling + 1e-12
    ci_low, ci_high = cluster_delta_interval(
        per_query["Recall@50"], variant_per_query["Recall@50"], validation["search_query"])
    active_fields = [field for field, weight in weights.items() if weight]
    row = {
        **experiment, "experiment": name,
        "config": json.dumps({**BM25_CONFIG, "field_weights": weights}, sort_keys=True),
        "fields": " + ".join(active_fields),
        **{metric: float(value) for metric, value in variant_metrics.items()},
        "delta_Recall@50": variant_metrics["Recall@50"] - metrics["Recall@50"],
        "delta_ci_low": ci_low, "delta_ci_high": ci_high,
        "bootstrap_confidence": BOOTSTRAP_CONFIDENCE, "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
        "index_seconds": sum(field_index_seconds[field] for field in active_fields),
        "index_MiB": sum(field_index_bytes[field] for field in active_fields) / 2**20,
        "search_seconds": variant_search_seconds,
        "search_p50_ms": float(np.percentile(variant_query_seconds, 50) * 1000),
        "search_p95_ms": float(np.percentile(variant_query_seconds, 95) * 1000),
        "rss_MiB": psutil.Process().memory_info().rss / 2**20,
        "empty_query_texts": sum(not ids for ids in variant_predictions.values()),
        "conclusion": f"Recall@50={variant_metrics['Recall@50']:.4%}; веса полей={weights}",
    }
    e1_rows.append(row)
    e1_per_query[name] = variant_per_query
    if variant_metrics["Recall@50"] > best_e1_recall:
        best_e1_experiment, best_e1_recall = name, variant_metrics["Recall@50"]
        best_e1_predictions = variant_predictions
    print(f"{name}: Recall@50={variant_metrics['Recall@50']:.4%}")

experiments = pd.concat([
    experiments.loc[~experiments.experiment.isin(E1_VARIANTS)], pd.DataFrame(e1_rows)
], ignore_index=True)
assert experiments.experiment.is_unique
experiments.to_csv(EXPERIMENTS_PATH, index=False)
comparison = experiments.loc[experiments.experiment.isin(["E0", *E1_VARIANTS])].copy()
display(comparison[["experiment", "Recall@1", "Recall@5", "Recall@10", "Recall@20", "Recall@50",
                    "available_Recall@50", "delta_Recall@50", "delta_ci_low", "delta_ci_high"]].round(6))
display(comparison[["experiment", "index_seconds", "search_seconds", "index_MiB"]].round(2))
best_experiment = best_e1_experiment if best_e1_recall > metrics["Recall@50"] else "E0"
print(f"Лучший вариант E1: {best_e1_experiment} — {best_e1_recall:.4%}")
print(f"Лучший вариант с учётом E0: {best_experiment}")

E1_title: Recall@50=1.4627%


E1_description: Recall@50=1.6294%


E1_title1_description1: Recall@50=1.8960%


E1_title2_description1: Recall@50=1.8315%


E1_title4_description1: Recall@50=1.7560%


,experiment,Recall@1,Recall@5,Recall@10,Recall@20,Recall@50,available_Recall@50,delta_Recall@50,delta_ci_low,delta_ci_high
2,E0,0.001626,0.005165,0.008089,0.012069,0.019168,0.297620,0.000000,NaN,NaN
3,E1_title,0.001385,0.004453,0.006679,0.009545,0.014627,0.233371,-0.004541,-0.005846,-0.003337
4,E1_description,0.001275,0.004290,0.007144,0.010456,0.016294,0.253143,-0.002873,-0.003556,-0.002169
5,E1_title1_description1,0.001782,0.005579,0.008227,0.012068,0.018960,0.298519,-0.000207,-0.000954,0.000481
6,E1_title2_description1,0.001769,0.005324,0.008113,0.011961,0.018315,0.290581,-0.000853,-0.001760,0.000089
7,E1_title4_description1,0.001545,0.005050,0.007639,0.011418,0.017560,0.278194,-0.001607,-0.002670,-0.000536


,experiment,index_seconds,search_seconds,index_MiB
2,E0,21.61,3.52,183.35
3,E1_title,1.51,2.28,6.86
4,E1_description,21.29,3.95,181.27
5,E1_title1_description1,22.79,4.78,188.13
6,E1_title2_description1,22.79,4.77,188.13
7,E1_title4_description1,22.79,4.77,188.13


Лучший вариант E1: E1_title1_description1 — 1.8960%
Лучший вариант с учётом E0: E0


Время поиска измерено отдельно для каждой конфигурации, включая токенизацию запросов.
Для суммы полей стоимость индексации — сумма двух индексов. Индексы между вариантами
переиспользуются; RSS включает все загруженные индексы и результаты.

In [22]:
delta = e1_per_query[best_e1_experiment]["Recall@50"] - per_query["Recall@50"]
display(pd.Series({
    "Группы с ростом Recall@50": int(delta.gt(0).sum()),
    "Группы со снижением Recall@50": int(delta.lt(0).sum()),
    "Без изменения": int(delta.eq(0).sum()),
}, name=best_e1_experiment).to_frame())

# Первые три различных текста каждого типа в алфавитном порядке.
examples = []
titles_by_id = items.set_index("item_id")["item_title_raw"]
for label, improved in [("Найден только E1", True), ("Найден только E0", False)]:
    seen_texts = set()
    for query in validation.sort_values("search_query", kind="stable").itertuples():
        if query.search_query in seen_texts:
            continue
        base = predictions[query.search_query]
        candidate = best_e1_predictions[query.search_query]
        difference = (set(candidate) - set(base)) if improved else (set(base) - set(candidate))
        changed_positives = sorted(query.available & difference)
        if not changed_positives:
            continue
        item_id = changed_positives[0]
        examples.append({
            "Случай": label, "Запрос": query.search_query, "group_id": query.Index,
            "item_id": item_id, "Заголовок позитива": titles_by_id[item_id],
            "Ранг E0": base.index(item_id) + 1 if item_id in base else ">50",
            "Ранг E1": candidate.index(item_id) + 1 if item_id in candidate else ">50",
        })
        seen_texts.add(query.search_query)
        if len(seen_texts) == 3:
            break
display(pd.DataFrame(examples))

,E1_title1_description1
Группы с ростом Recall@50,262
Группы со снижением Recall@50,264
Без изменения,67855


,Случай,Запрос,group_id,item_id,Заголовок позитива,Ранг E0,Ранг E1
0,Найден только E1,3d печать,412,6cb949d70341a638,3D печать,>50,26
1,Найден только E1,volkswagen polo,2835,20277fdaf89ff58f,"Volkswagen Polo 2015 Аренда авто с выкупом, лизинг",>50,15
2,Найден только E1,аварийное вскрытие замков двери 24/7,3190,d86a5c4ef8197e64,Аварийное вскрытие замков двери 24/7,>50,3
3,Найден только E0,3d печать,361,de068bbdaf44201d,Печать на 3D принтере (3Д моделирование),11,>50
4,Найден только E0,sup аренда,2710,1e38849ba62257fb,"Лучшие сапы в Пензе,сапборд в аренду,сап в аренду",22,>50
5,Найден только E0,автовоз из владивостока,5099,4c1e821887b89a0a,Услуги автовоза-доставка автомобилей по России,43,>50


Лучший E1 — сумма полей 1:1: **Recall@50 = 1,896%**, против **1,917%** у E0.
Разница — **−0,021 п.п.**, 99% bootstrap-интервал: **[−0,095; +0,048] п.п.**
Улучшение не подтверждено. Усиление заголовка до 2:1 и 4:1 снижает Recall@50.

Сумма 1:1 немного улучшает Recall@1/5/10, но не основную метрику.
Для суммы полей нужны два индекса. Сохраняем E0: у него наибольший Recall@50
на этом split и меньшее время поиска.

## 11. E2 — TF-IDF

Word TF-IDF учитывает слова и соседние пары слов; character TF-IDF — фрагменты слов,
которые могут совпасть при опечатках и изменении окончаний.
Оба варианта используют заголовок и описание, L2-нормализацию и cosine similarity.

Словарь и IDF считаем только по корпусу. Редкие n-граммы (`df < 2`) исключаем;
лимит словаря — 300 тыс. для word и 200 тыс. для char, чтобы ограничить память.
TF — `1 + log(count)`. Стоп-слова сохраняем. Параметры фиксированы без подбора.

Поиск точный, батчами на CPU. При равных оценках выбираем меньший item_id;
нулевые совпадения не включаем в выдачу.

In [23]:
def sparse_top_k(query_matrix, documents_t, k=50, batch_size=TFIDF_BATCH_SIZE, n_threads=TFIDF_THREADS):
    results, tie_rows = [], 0
    if k <= 0 or documents_t.shape[1] == 0:
        return [np.empty(0, dtype=np.int64) for _ in range(query_matrix.shape[0])], 0
    for start in range(0, query_matrix.shape[0], batch_size):
        query_batch = query_matrix[start:start + batch_size]
        matches = sp_matmul_topn(query_batch, documents_t, top_n=min(k + 1, documents_t.shape[1]),
                                threshold=0.0, sort=True, n_threads=n_threads)
        for i in range(query_batch.shape[0]):
            row = matches.getrow(i)
            positive = row.data > 0
            ids, scores = row.indices[positive], row.data[positive]
            order = np.lexsort((ids, -scores))
            # k+1 позволяет заметить равенство на границе top-k.
            if len(order) > k and scores[order[k - 1]] == scores[order[k]]:
                full_scores = (query_batch.getrow(i) @ documents_t).toarray().ravel()
                results.append(top_k(full_scores, k))
                tie_rows += 1
            else:
                results.append(ids[order[:k]])
    return results, tie_rows


rng = np.random.default_rng(SEED)
toy_docs = csr_matrix(rng.integers(0, 3, (70, 12)).astype(np.float32))
toy_queries = csr_matrix(rng.integers(0, 3, (6, 12)).astype(np.float32))
for threads in (1, TFIDF_THREADS):
    found, _ = sparse_top_k(toy_queries, toy_docs.T.tocsr(), k=5, batch_size=2, n_threads=threads)
    expected = (toy_queries @ toy_docs.T).toarray()
    for i in range(len(found)):
        np.testing.assert_array_equal(found[i], top_k(expected[i], 5))
identical_docs = csr_matrix(np.ones((60, 2), dtype=np.float32))
found, ties = sparse_top_k(csr_matrix([[1., 1.], [0., 0.]], dtype=np.float32), identical_docs.T.tocsr())
assert found[0].tolist() == list(range(50)) and found[1].tolist() == [] and ties == 1

# Искусственные примеры проверяют токенизацию, а не качество на validation.
toy_texts = ["ремонт холодильников", "химчистка диванов", "ремонт iphone 13", "чистка кондиционеров"]
toy_query_texts = ["холодилник", "диван", "iphone13", "кондиционер"]
toy_vectorizer = TfidfVectorizer(**TFIDF_COMMON, analyzer="char_wb", ngram_range=(3, 5))
toy_matrix = toy_vectorizer.fit_transform(toy_texts)
toy_q = toy_vectorizer.transform(toy_query_texts)
found, _ = sparse_top_k(toy_q, toy_matrix.T.tocsr(), k=1)
assert [int(ids[0]) for ids in found] == [0, 1, 2, 3]
np.testing.assert_allclose(np.asarray(toy_matrix.multiply(toy_matrix).sum(axis=1)).ravel(), 1, atol=1e-6)
assert toy_vectorizer.transform([""]).nnz == 0
print("Разреженный top-k, равенства, пустые запросы и токенизация: проверки пройдены")

Разреженный top-k, равенства, пустые запросы и токенизация: проверки пройдены


In [24]:
tfidf_models, tfidf_index_rows = {}, []
for name, config in TFIDF_VARIANTS.items():
    start = perf_counter()
    vectorizer = TfidfVectorizer(**TFIDF_COMMON, **config)
    matrix = vectorizer.fit_transform(corpus_text).tocsr()
    documents_t = matrix.T.tocsr()
    duration = perf_counter() - start
    assert matrix.shape[0] == len(item_ids) and matrix.dtype == np.float32
    norms = np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel()
    np.testing.assert_allclose(norms[norms > 0], 1, atol=2e-5)
    index_bytes_tfidf = sum(a.nbytes for m in (matrix, documents_t) for a in (m.data, m.indices, m.indptr))
    tfidf_models[name] = (vectorizer, documents_t)
    tfidf_index_rows.append({
        "experiment": name, "index_seconds": duration, "features": matrix.shape[1],
        "nonzero_values": matrix.nnz, "empty_documents": int((norms == 0).sum()),
        "index_MiB": index_bytes_tfidf / 2**20,
    })
    print(f"{name}: {matrix.shape[1]:,} признаков; {duration:.1f} с")
    del matrix, norms
    _ = gc.collect()
tfidf_index_info = pd.DataFrame(tfidf_index_rows).set_index("experiment")
display(tfidf_index_info.round(2))

E2_word: 300,000 признаков; 45.2 с


E2_char: 200,000 признаков; 131.4 с


,index_seconds,features,nonzero_values,empty_documents,index_MiB
experiment,,,,,
E2_word,45.18,300000,40294947,2,616.72
E2_char,131.41,200000,319117616,0,4870.83


Время индексации включает построение TF-IDF и транспонирование.
Размер матриц указан до удаления исходной CSR; словарь Python в него не входит.
Время поиска включает преобразование запросов и извлечение top-50.
97,5% bootstrap-интервалы рассчитаны для двух сравнений с E0 (Бонферрони), по текстам запросов.
TF-IDF использует до 8 CPU-потоков; BM25 измерен последовательно. Время отражает эти режимы запуска.

In [25]:
e2_rows, e2_predictions, e2_per_query = [], {}, {}
for name, (vectorizer, documents_t) in tfidf_models.items():
    start = perf_counter()
    query_matrix = vectorizer.transform(val_texts).tocsr()
    positions, tie_rows = sparse_top_k(query_matrix, documents_t, max(RECALL_K))
    candidate_predictions = {text: item_ids[ids].tolist() for text, ids in zip(val_texts, positions)}
    duration = perf_counter() - start
    assert set(candidate_predictions) == set(val_texts)
    assert all(len(ids) <= 50 and len(ids) == len(set(ids)) and set(ids) <= corpus_ids
               for ids in candidate_predictions.values())
    candidate_per_query = evaluate(validation, candidate_predictions)
    assert candidate_per_query.index.equals(per_query.index)
    assert candidate_per_query["Recall@50"].notna().all()
    assert candidate_per_query[[f"Recall@{k}" for k in RECALL_K]].diff(axis=1).iloc[:, 1:].ge(0).all().all()
    candidate_metrics = candidate_per_query.mean()
    assert candidate_metrics["Recall@50"] <= ceiling + 1e-12
    ci_low, ci_high = cluster_delta_interval(per_query["Recall@50"], candidate_per_query["Recall@50"],
                                             validation["search_query"], confidence=E2_CONFIDENCE)
    config = {**TFIDF_COMMON, **TFIDF_VARIANTS[name], "dtype": "float32",
              "batch_size": TFIDF_BATCH_SIZE, "threads": TFIDF_THREADS}
    row = {
        **experiment, "experiment": name, "config": json.dumps(config, sort_keys=True),
        "bm25s_version": None, "sklearn_version": importlib.metadata.version("scikit-learn"),
        "sparse_dot_topn_version": importlib.metadata.version("sparse-dot-topn"),
        "token_pattern": TOKEN_PATTERN if name == "E2_word" else None,
        **{metric: float(value) for metric, value in candidate_metrics.items()},
        **tfidf_index_info.loc[name].to_dict(), "search_seconds": duration,
        "delta_Recall@50": candidate_metrics["Recall@50"] - metrics["Recall@50"],
        "delta_ci_low": ci_low, "delta_ci_high": ci_high,
        "bootstrap_confidence": E2_CONFIDENCE, "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
        "rss_MiB": psutil.Process().memory_info().rss / 2**20,
        "empty_query_texts": sum(not ids for ids in candidate_predictions.values()),
        "boundary_tie_queries": tie_rows,
        "conclusion": f"Recall@50={candidate_metrics['Recall@50']:.4%}; {TFIDF_VARIANTS[name]['analyzer']} TF-IDF",
    }
    e2_rows.append(row)
    e2_predictions[name] = candidate_predictions
    e2_per_query[name] = candidate_per_query
    print(f"{name}: Recall@50={candidate_metrics['Recall@50']:.4%}; поиск {duration:.1f} с")

experiments = pd.concat([experiments.loc[~experiments.experiment.isin(TFIDF_VARIANTS)], pd.DataFrame(e2_rows)], ignore_index=True)
assert experiments.experiment.is_unique
experiments.to_csv(EXPERIMENTS_PATH, index=False)
display(experiments[["experiment", "Recall@1", "Recall@5", "Recall@10", "Recall@20", "Recall@50",
                     "available_Recall@50", "delta_Recall@50"]].round(6))
display(experiments.loc[experiments.experiment.isin(TFIDF_VARIANTS),
    ["experiment", "delta_ci_low", "delta_ci_high", "index_seconds", "search_seconds", "index_MiB", "empty_query_texts", "boundary_tie_queries"]].round(4))

E2_word: Recall@50=1.5504%; поиск 1.8 с


E2_char: Recall@50=1.3516%; поиск 12.4 с


,experiment,Recall@1,Recall@5,Recall@10,Recall@20,Recall@50,available_Recall@50,delta_Recall@50
0,E0,0.001626,0.005165,0.008089,0.012069,0.019168,0.297620,0.000000
1,E1_title,0.001385,0.004453,0.006679,0.009545,0.014627,0.233371,-0.004541
2,E1_description,0.001275,0.004290,0.007144,0.010456,0.016294,0.253143,-0.002873
3,E1_title1_description1,0.001782,0.005579,0.008227,0.012068,0.018960,0.298519,-0.000207
4,E1_title2_description1,0.001769,0.005324,0.008113,0.011961,0.018315,0.290581,-0.000853
5,E1_title4_description1,0.001545,0.005050,0.007639,0.011418,0.017560,0.278194,-0.001607
6,E2_word,0.000755,0.002981,0.005523,0.009084,0.015504,0.239996,-0.003664
7,E2_char,0.000714,0.002582,0.004389,0.007419,0.013516,0.204643,-0.005652


,experiment,delta_ci_low,delta_ci_high,index_seconds,search_seconds,index_MiB,empty_query_texts,boundary_tie_queries
6,E2_word,-0.0045,-0.0029,45.1764,1.8135,616.7183,196.0,758.0
7,E2_char,-0.0067,-0.0047,131.4104,12.4355,4870.8331,1.0,713.0


### Какие позитивы находят методы

Считаем уникальные размеченные пары запрос–объявление, включая недоступные в корпусе.
Пары имеют равный вес в этой таблице; основная метрика выше усреднена по запросам.

In [26]:
pair_rows, e2_changed_examples = [], []
for name, candidate_predictions in e2_predictions.items():
    counts = {"Оба": 0, "Только E0": 0, "Только TF-IDF": 0, "Ни один": 0}
    example_counts = {"Только E0": set(), "Только TF-IDF": set()}
    for query in validation.sort_values("search_query", kind="stable").itertuples():
        base, candidate = predictions[query.search_query], candidate_predictions[query.search_query]
        base_found, candidate_found = query.positives & set(base), query.positives & set(candidate)
        counts["Оба"] += len(base_found & candidate_found)
        counts["Только E0"] += len(base_found - candidate_found)
        counts["Только TF-IDF"] += len(candidate_found - base_found)
        counts["Ни один"] += len(query.positives - (base_found | candidate_found))
        for label, changed in [("Только E0", base_found - candidate_found), ("Только TF-IDF", candidate_found - base_found)]:
            if changed and len(example_counts[label]) < 5 and query.search_query not in example_counts[label]:
                item_id = sorted(changed)[0]
                e2_changed_examples.append({
                    "Метод": name, "Случай": label, "Запрос": query.search_query, "group_id": query.Index,
                    "item_id": item_id, "Заголовок": titles_by_id[item_id],
                    "Ранг E0": base.index(item_id) + 1 if item_id in base else ">50",
                    "Ранг TF-IDF": candidate.index(item_id) + 1 if item_id in candidate else ">50",
                })
                example_counts[label].add(query.search_query)
    assert sum(counts.values()) == validation.n_positives.sum()
    pair_rows.append({"Метод": name, **counts, "Всего пар": sum(counts.values())})
display(pd.DataFrame(pair_rows).set_index("Метод"))
# Примеры выбираются по алфавиту, без отбора по величине улучшения.
display(pd.DataFrame(e2_changed_examples))

,Оба,Только E0,Только TF-IDF,Ни один,Всего пар
Метод,,,,,
E2_word,1143,512,184,85348,87187
E2_char,882,773,235,85297,87187


,Метод,Случай,Запрос,group_id,item_id,Заголовок,Ранг E0,Ранг TF-IDF
0,E2_word,Только E0,3d печать,362,b2b8ad0d0f44ca87,"3D Печать, 3D Сканирование, 3D Моделирование 24/7",42,>50
1,E2_word,Только TF-IDF,3d печать,399,5496b03fc45256c7,3д/3D печать,>50,38
2,E2_word,Только E0,mercari выкуп,2071,fab5bf956436a174,Байер Япония (Mercari/Yahoo/Rakuten и тд),1,>50
3,E2_word,Только E0,smas лифтинг,2527,6fce025768d766a2,Микроигольчатый РФ лифтинг. Лицо+Шея. Ищу моделей,42,>50
4,E2_word,Только TF-IDF,volkswagen polo,2835,20277fdaf89ff58f,"Volkswagen Polo 2015 Аренда авто с выкупом, лизинг",>50,36
5,E2_word,Только E0,автовоз из владивостока,5099,4c1e821887b89a0a,Услуги автовоза-доставка автомобилей по России,43,>50
6,E2_word,Только TF-IDF,автовоз москва,5164,aa6a7171651854df,Автовоз в/из Екатеринбург. Перевозка автомобилей,>50,49
7,E2_word,Только TF-IDF,автовыкуп,5257,94afaadfef830f43,Автовыкуп Липецк. Выкуп авто сегодня. Продать авто,>50,18
8,E2_word,Только E0,автоэлектрик круглосуточно,10083,9328d4b848a5a3d6,Автоэлектрик,22,>50
9,E2_word,Только TF-IDF,активация vr очки,11880,86b708dd82f85a62,"Настройка Oculus Quest 2,3,3s VR шлем",>50,37


Word TF-IDF получил **Recall@50 = 1,550%**, character — **1,352%**,
BM25 — **1,917%**. Интервалы разницы с E0 целиком ниже нуля.
Оба TF-IDF уступают BM25 как самостоятельные методы отбора.

При этом word нашёл 184 позитивные пары вне top-50 BM25, char — 235.
Это дополняемость выдач, а не доказательство улучшения гибрида.

Для «автоэвакуатор» char нашёл позитив на 4-м месте: в описании — «услуги автоэвакуатора».
Для «ip видеонаблюдение» — на 9-м: в объявлении используется форма «видеонаблюдения».
Оба позитива отсутствуют в top-50 BM25. Обратный пример — «mercari выкуп»:
BM25 поставил позитив первым, оба TF-IDF не включили его в top-50.
Основным lexical baseline остаётся BM25.

## 12. E3 — zero-shot dense retrieval

Гипотеза: семантические embeddings найдут позитивы без точного совпадения слов.
Одна модель `intfloat/multilingual-e5-small`, без обучения и подбора параметров.
Заголовок и описание сохраняют регистр; для каждого нормализованного запроса берём
лексикографически первый исходный текст из train, без обращения к позитивам.

384 измерения, mean pooling с маской padding, L2-нормализация, cosine similarity.
Префиксы `query: ` и `passage: `, максимум 512 токенов с обрезкой справа.
Точный поиск по всему корпусу в float32; равенства разрешаем по item_id.
Отрицательные и нулевые оценки допустимы. Контексты и знаменатель Recall прежние.
Для разницы с E0 — парный bootstrap по текстам, 2000 повторов, 95% интервал.

In [27]:
from sentence_transformers import SentenceTransformer

DENSE_CONFIG = {
    "model": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "dimension": 384, "pooling": "mean", "normalize_embeddings": True,
    "similarity": "cosine", "dtype": "float32", "max_length": 512,
    "query_prefix": "query: ", "document_prefix": "passage: ",
    "batch_size": 32, "search_batch_size": 64,
    "query_text": "lexicographically first raw text per normalized text",
    "document_text": "raw title + newline + raw description",
    "tie_break": "item_id ascending", "search": "exact matrix multiplication",
}
DENSE_DIR = Path(".cache/e3")
DENSE_DIR.mkdir(parents=True, exist_ok=True)
# Отключаем TF32: поиск и encoder используют float32 без autocast.
torch.set_float32_matmul_precision("highest")
torch.set_num_threads(min(8, os.cpu_count() or 1))
start = perf_counter()
encoder = SentenceTransformer(
    DENSE_CONFIG["model"], revision=DENSE_CONFIG["revision"],
    cache_folder=".cache/huggingface", device=str(DEVICE),
    local_files_only=True, trust_remote_code=False,
)
encoder.float().eval()
encoder.max_seq_length = DENSE_CONFIG["max_length"]
encoder.tokenizer.truncation_side = "right"
assert encoder.get_embedding_dimension() == 384
assert encoder[1].pooling_mode == "mean"
assert encoder[1].include_prompt
model_load_seconds = perf_counter() - start
print(f"Модель: {DENSE_CONFIG['model']}; revision: {DENSE_CONFIG['revision']}")
print(f"Устройство: {DEVICE}; загрузка модели: {model_load_seconds:.2f} с")
print(encoder)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24925.09it/s]

Модель: intfloat/multilingual-e5-small; revision: 614241f622f53c4eeff9890bdc4f31cfecc418b3
Устройство: cuda; загрузка модели: 1.31 с
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


In [28]:
def synchronize_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


@torch.inference_mode()
def dense_top_k(query_vectors, document_vectors, k=50, device=DEVICE, batch_size=64):
    if k < 1 or batch_size < 1 or len(document_vectors) < 1:
        raise ValueError("Нужны положительные k, batch_size и непустой корпус")
    if query_vectors.ndim != 2 or document_vectors.ndim != 2 or query_vectors.shape[1] != document_vectors.shape[1]:
        raise ValueError("Несовместимые размерности embeddings")
    if not np.isfinite(query_vectors).all() or not np.isfinite(document_vectors).all():
        raise ValueError("Embeddings содержат NaN/inf")
    documents = torch.as_tensor(document_vectors, dtype=torch.float32, device=device)
    k = min(k, len(documents))
    positions, boundary_ties = [], 0
    for begin in range(0, len(query_vectors), batch_size):
        queries_batch = torch.as_tensor(query_vectors[begin:begin + batch_size], dtype=torch.float32, device=device)
        scores = queries_batch @ documents.T
        values, indices = torch.topk(scores, min(k + 1, len(documents)), dim=1, sorted=True)
        values, indices = values.cpu().numpy(), indices.cpu().numpy()
        for row in range(len(values)):
            if k < len(documents) and values[row, k - 1] == values[row, k]:
                # topk произвольно выбирает равные элементы: восстанавливаем всю границу.
                full = scores[row].cpu().numpy()
                threshold = values[row, k - 1]
                higher = np.flatnonzero(full > threshold)
                tied = np.flatnonzero(full == threshold)[:k - len(higher)]
                chosen = np.concatenate([higher, tied])
                chosen = chosen[np.lexsort((chosen, -full[chosen]))]
                boundary_ties += 1
            else:
                chosen = indices[row, :k]
                chosen = chosen[np.lexsort((chosen, -values[row, :k]))]
            positions.append(chosen)
    return positions, boundary_ties


# Дубли, нулевые и отрицательные оценки, k больше корпуса, разные батчи и ID mapping.
rng = np.random.default_rng(SEED)
test_documents = rng.integers(-2, 3, size=(73, 8)).astype(np.float32)
test_documents[10:65] = test_documents[0]
test_queries = np.vstack([rng.integers(-2, 3, size=(7, 8)), np.zeros((1, 8))]).astype(np.float32)
full_scores = test_queries @ test_documents.T
test_devices = [torch.device("cpu")] + ([DEVICE] if DEVICE.type != "cpu" else [])
for test_device in test_devices:
    for k in (1, 50, 100):
        expected = [np.lexsort((np.arange(len(row)), -row))[:k] for row in full_scores]
        for batch_size in (1, 3, 64):
            actual, _ = dense_top_k(test_queries, test_documents, k, test_device, batch_size)
            for found, reference in zip(actual, expected):
                np.testing.assert_array_equal(found, reference)
    empty, _ = dense_top_k(np.empty((0, 8), dtype=np.float32), test_documents, device=test_device)
    assert empty == []
    negative, _ = dense_top_k(np.ones((1, 2), dtype=np.float32), -np.ones((60, 2), dtype=np.float32), device=test_device)
    np.testing.assert_array_equal(negative[0], np.arange(50))
print("Exact top-k: проверки пройдены на", ", ".join(map(str, test_devices)))

Exact top-k: проверки пройдены на cpu, cuda


In [29]:
start = perf_counter()
dense_documents = (items.item_title_raw.fillna("") + "\n" + items.item_description_raw.fillna("")).tolist()
raw_queries = pd.read_parquet(DATA_DIR / "train.parquet", columns=["search_query"])
raw_queries["normalized"] = normalize_text(raw_queries.search_query)
raw_query_map = raw_queries.sort_values("search_query").drop_duplicates("normalized").set_index("normalized").search_query
dense_queries = raw_query_map.loc[val_texts].fillna("").tolist()
assert normalize_text(pd.Series(dense_queries)).tolist() == val_texts
del raw_queries, raw_query_map
synchronize_device(DEVICE)
corpus_embeddings = encoder.encode(
    dense_documents, prompt=DENSE_CONFIG["document_prefix"],
    batch_size=DENSE_CONFIG["batch_size"], normalize_embeddings=True, show_progress_bar=True,
)
synchronize_device(DEVICE)
dense_index_seconds = perf_counter() - start
start = perf_counter()
query_embeddings = encoder.encode(
    dense_queries, prompt=DENSE_CONFIG["query_prefix"],
    batch_size=DENSE_CONFIG["batch_size"], normalize_embeddings=True, show_progress_bar=True,
)
synchronize_device(DEVICE)
dense_query_encode_seconds = perf_counter() - start
for vectors, count in [(corpus_embeddings, len(item_ids)), (query_embeddings, len(val_texts))]:
    assert vectors.shape == (count, 384) and vectors.dtype == np.float32
    assert np.isfinite(vectors).all()
    np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=2e-5)
start = perf_counter()
np.save(DENSE_DIR / "corpus.npy", corpus_embeddings, allow_pickle=False)
np.save(DENSE_DIR / "queries.npy", query_embeddings, allow_pickle=False)
pd.DataFrame({"row": np.arange(len(item_ids)), "item_id": item_ids}).to_csv(DENSE_DIR / "items.csv", index=False)
pd.DataFrame({"row": np.arange(len(val_texts)), "text_id": [text_ids[t] for t in val_texts]}).to_csv(DENSE_DIR / "queries.csv", index=False)
dense_metadata = {
    "config": DENSE_CONFIG, "device": str(DEVICE), "split_sha256": SPLIT_HASH,
    "corpus_sha256": experiment["corpus_sha256"], "train_sha256": experiment["train_sha256"],
    "versions": {name: importlib.metadata.version(name) for name in
                 ["sentence-transformers", "transformers", "tokenizers", "huggingface-hub", "torch"]},
    "files": {name: sha256(DENSE_DIR / name) for name in ["corpus.npy", "queries.npy", "items.csv", "queries.csv"]},
}
(DENSE_DIR / "metadata.json").write_text(json.dumps(dense_metadata, ensure_ascii=False, indent=2))
dense_save_seconds = perf_counter() - start
np.testing.assert_array_equal(pd.read_csv(DENSE_DIR / "items.csv", dtype={"item_id": str}).item_id, item_ids)
np.testing.assert_array_equal(np.load(DENSE_DIR / "corpus.npy", mmap_mode="r"), corpus_embeddings)
display(pd.Series({
    "Документы": len(corpus_embeddings), "Тексты запросов": len(query_embeddings),
    "Кодирование корпуса, с": dense_index_seconds, "Кодирование запросов, с": dense_query_encode_seconds,
    "Сохранение и контрольные суммы, с": dense_save_seconds,
    "Документов/с": len(item_ids) / dense_index_seconds,
    "Embeddings корпуса, МиБ": corpus_embeddings.nbytes / 2**20,
}, name="Значение").to_frame())

Batches:   0%|          | 0/5913 [00:00<?, ?it/s]

Batches:   0%|          | 2/5913 [00:00<07:03, 13.97it/s]

Batches:   0%|          | 5/5913 [00:00<04:54, 20.03it/s]

Batches:   0%|          | 8/5913 [00:00<04:28, 21.97it/s]

Batches:   0%|          | 11/5913 [00:00<04:18, 22.85it/s]

...

Batches:  84%|████████▎ | 386/462 [00:01<00:00, 324.59it/s]

Batches:  91%|█████████ | 419/462 [00:01<00:00, 322.61it/s]

Batches:  98%|█████████▊| 453/462 [00:01<00:00, 326.28it/s]

Batches: 100%|██████████| 462/462 [00:01<00:00, 317.98it/s]

,Значение
Документы,189212.000000
Тексты запросов,14781.000000
"Кодирование корпуса, с",151.445759
"Кодирование запросов, с",1.482111
"Сохранение и контрольные суммы, с",0.233606
Документов/с,1249.371402
"Embeddings корпуса, МиБ",277.166016


In [30]:
# Полный расчёт на том же устройстве проверяет exact top-k без различий BLAS.
reference_scores = query_embeddings[:8] @ corpus_embeddings[:1000].T
for test_device in test_devices:
    with torch.inference_mode():
        device_scores = (torch.as_tensor(query_embeddings[:8], device=test_device)
                         @ torch.as_tensor(corpus_embeddings[:1000], device=test_device).T).cpu().numpy()
    np.testing.assert_allclose(device_scores, reference_scores, atol=2e-6, rtol=2e-6)
    reference = [np.lexsort((np.arange(len(row)), -row))[:50] for row in device_scores]
    found, _ = dense_top_k(query_embeddings[:8], corpus_embeddings[:1000], device=test_device)
    for actual, expected in zip(found, reference):
        np.testing.assert_array_equal(actual, expected)
start = perf_counter()
dense_positions, dense_ties = dense_top_k(
    query_embeddings, corpus_embeddings, batch_size=DENSE_CONFIG["search_batch_size"])
synchronize_device(DEVICE)
dense_search_seconds = perf_counter() - start
dense_predictions = {text: item_ids[positions].tolist() for text, positions in zip(val_texts, dense_positions)}
assert set(dense_predictions) == set(val_texts)
assert all(len(ids) == 50 and len(set(ids)) == 50 and set(ids) <= corpus_ids for ids in dense_predictions.values())
dense_per_query = evaluate(validation, dense_predictions)
assert dense_per_query.index.equals(per_query.index)
assert dense_per_query["Recall@50"].notna().all()
assert dense_per_query[[f"Recall@{k}" for k in RECALL_K]].diff(axis=1).iloc[:, 1:].ge(0).all().all()
dense_metrics = dense_per_query.mean()
assert dense_metrics["Recall@50"] <= ceiling + 1e-12
dense_ci = cluster_delta_interval(per_query["Recall@50"], dense_per_query["Recall@50"], validation.search_query, confidence=.95)
dense_row = {
    **experiment, "experiment": "E3", "config": json.dumps(DENSE_CONFIG, sort_keys=True),
    "token_pattern": None, "bm25s_version": None, "device": str(DEVICE),
    **{name: float(value) for name, value in dense_metrics.items()},
    "index_seconds": dense_index_seconds, "search_seconds": dense_search_seconds + dense_query_encode_seconds,
    "query_encode_seconds": dense_query_encode_seconds, "exact_search_seconds": dense_search_seconds,
    "model_load_seconds": model_load_seconds, "save_seconds": dense_save_seconds,
    "index_MiB": corpus_embeddings.nbytes / 2**20,
    "delta_Recall@50": dense_metrics["Recall@50"] - metrics["Recall@50"],
    "delta_ci_low": dense_ci[0], "delta_ci_high": dense_ci[1],
    "bootstrap_confidence": .95, "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "boundary_tie_queries": dense_ties, "empty_query_texts": 0,
    "sentence_transformers_version": importlib.metadata.version("sentence-transformers"),
    "transformers_version": importlib.metadata.version("transformers"),
    "rss_MiB": psutil.Process().memory_info().rss / 2**20,
    "conclusion": f"Recall@50={dense_metrics['Recall@50']:.4%}; zero-shot multilingual-e5-small",
}
experiments = pd.concat([experiments.loc[experiments.experiment.ne("E3")], pd.DataFrame([dense_row])], ignore_index=True)
assert experiments.experiment.is_unique
experiments.to_csv(EXPERIMENTS_PATH, index=False)
display(experiments[["experiment", "Recall@1", "Recall@5", "Recall@10", "Recall@20", "Recall@50", "available_Recall@50"]].round(6))
display(pd.Series({"Разница с E0, п.п.": 100 * dense_row["delta_Recall@50"],
                   "95% интервал, нижняя граница, п.п.": 100 * dense_ci[0],
                   "95% интервал, верхняя граница, п.п.": 100 * dense_ci[1],
                   "Exact search, с": dense_search_seconds,
                   "Равенства на границе top-50": dense_ties}, name="Значение").to_frame())

,experiment,Recall@1,Recall@5,Recall@10,Recall@20,Recall@50,available_Recall@50
0,E0,0.001626,0.005165,0.008089,0.012069,0.019168,0.297620
1,E1_title,0.001385,0.004453,0.006679,0.009545,0.014627,0.233371
2,E1_description,0.001275,0.004290,0.007144,0.010456,0.016294,0.253143
3,E1_title1_description1,0.001782,0.005579,0.008227,0.012068,0.018960,0.298519
4,E1_title2_description1,0.001769,0.005324,0.008113,0.011961,0.018315,0.290581
5,E1_title4_description1,0.001545,0.005050,0.007639,0.011418,0.017560,0.278194
6,E2_word,0.000755,0.002981,0.005523,0.009084,0.015504,0.239996
7,E2_char,0.000714,0.002582,0.004389,0.007419,0.013516,0.204643
8,E3,0.001363,0.004886,0.007467,0.011443,0.018395,0.287525


,Значение
"Разница с E0, п.п.",-0.077308
"95% интервал, нижняя граница, п.п.",-0.169459
"95% интервал, верхняя граница, п.п.",0.014792
"Exact search, с",0.352207
Равенства на границе top-50,1631.000000


### Дополняемость BM25 и dense

Уникальные позитивные пары при глубине 50, включая позитивы вне корпуса.
Это подсчёт пар, а не macro Recall. Примеры — первые пять различных текстов
по алфавиту в каждой категории различий; они не характеризуют частоту ошибок.

In [31]:
dense_pair_counts = {"Оба": 0, "Только BM25": 0, "Только dense": 0, "Ни один": 0}
dense_examples, seen_examples = [], {"Только BM25": set(), "Только dense": set()}
for query in validation.sort_values("search_query", kind="stable").itertuples():
    base, candidate = predictions[query.search_query], dense_predictions[query.search_query]
    base_found, dense_found = query.positives & set(base), query.positives & set(candidate)
    dense_pair_counts["Оба"] += len(base_found & dense_found)
    dense_pair_counts["Только BM25"] += len(base_found - dense_found)
    dense_pair_counts["Только dense"] += len(dense_found - base_found)
    dense_pair_counts["Ни один"] += len(query.positives - (base_found | dense_found))
    for label, changed in [("Только BM25", base_found - dense_found), ("Только dense", dense_found - base_found)]:
        if changed and len(seen_examples[label]) < 5 and query.search_query not in seen_examples[label]:
            item_id = sorted(changed)[0]
            dense_examples.append({"Случай": label, "Запрос": query.search_query, "group_id": query.Index,
                                   "item_id": item_id, "Заголовок": titles_by_id[item_id],
                                   "Ранг BM25": base.index(item_id) + 1 if item_id in base else ">50",
                                   "Ранг dense": candidate.index(item_id) + 1 if item_id in candidate else ">50"})
            seen_examples[label].add(query.search_query)
assert sum(dense_pair_counts.values()) == validation.n_positives.sum()
dense_pair_table = pd.Series(dense_pair_counts, name="Пары").to_frame()
dense_pair_table["Доля пар"] = dense_pair_table.Пары / dense_pair_table.Пары.sum()
display(dense_pair_table)
display(pd.DataFrame(dense_examples))

,Пары,Доля пар
Оба,954,0.010942
Только BM25,701,0.008040
Только dense,632,0.007249
Ни один,84900,0.973769


,Случай,Запрос,group_id,item_id,Заголовок,Ранг BM25,Ранг dense
0,Только BM25,3d печать,424,c9a823ae64fbd1eb,Профессиональная 3D печать,25,>50
1,Только dense,3d печать,412,6cb949d70341a638,3D печать,>50,16
2,Только BM25,777,888,d1f9f13c96294d41,Домашний интернет и тв в частный дом,6,>50
3,Только dense,ip видеонаблюдение,1466,960ee99868b1118f,Монтаж и обслуживание видеонаблюдения,>50,39
4,Только dense,iphone ломбард покупка,1481,cbe5f414681fd4da,Выкуп iPhone/MacBook/Samsung,>50,39
5,Только BM25,mercari выкуп,2071,fab5bf956436a174,Байер Япония (Mercari/Yahoo/Rakuten и тд),1,>50
6,Только BM25,sla печать,2512,b765441de5364699,"Художественная SLA печать,покрас 3D моделей",31,>50
7,Только BM25,smas лифтинг,2527,6fce025768d766a2,Микроигольчатый РФ лифтинг. Лицо+Шея. Ищу моделей,42,>50
8,Только dense,volkswagen polo,2835,20277fdaf89ff58f,"Volkswagen Polo 2015 Аренда авто с выкупом, лизинг",>50,14
9,Только dense,аварийное вскрытие замков двери 24/7,3190,d86a5c4ef8197e64,Аварийное вскрытие замков двери 24/7,>50,20


Dense получил **Recall@50 = 1,8395%**, BM25 — **1,9168%**.
Разница −0,0773 п.п.; 95% интервал [−0,1695; +0,0148] п.п. включает ноль.
Оснований заменять BM25 этим dense baseline нет.

Dense нашёл 632 позитивные пары вне top-50 BM25, но потерял 701 найденную BM25.
Оба метода нашли 954 пары. Знаменатель — 87 187 пар, включая недоступные в корпусе.
Дополняемость даёт основание проверить гибрид; сама по себе она не доказывает его пользу.

Для «iphone ломбард покупка» dense нашёл объявление «Выкуп iPhone/MacBook/Samsung»
на 39-м месте, BM25 — вне top-50. Для «mercari выкуп» BM25 нашёл позитив первым,
dense — вне top-50. Причины различий нельзя установить только по этим примерам.

На CUDA кодирование 189 212 объявлений заняло 151,45 с, 14 781 текста запроса — 1,48 с,
exact search — 0,35 с. Embeddings корпуса занимают 277,17 МиБ; это не полная память модели.
Время измерено для пакетной обработки и не является latency отдельного запроса.
Время CPU baseline и CUDA dense не сравнивает алгоритмы при равных ресурсах.